<a href="https://colab.research.google.com/github/Didarulisalmdidar/QLSTM_Multilayer_Network/blob/main/CrossSectorQLSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install pennylane

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 93.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 76.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 83.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 110.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 129.0 MB/s eta 0:00:00


In [ ]:
import os
import time
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import pennylane as qml
from torch.utils.data import DataLoader, TensorDataset

BASE_DIR   = "/content/drive/MyDrive/MSC Thesis"
DATA_DIR   = os.path.join(BASE_DIR, "Processed")
MODEL_DIR  = os.path.join(BASE_DIR, "CrossSectorModels_Dense")
LOSS_DIR   = os.path.join(BASE_DIR, "Losses_Dense")
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(LOSS_DIR,  exist_ok=True)

In [ ]:
# ── ↓↓↓ CHANGE THIS EACH SESSION ↓↓↓ ─────────────────────
SOURCE_SECTOR = "IT"
# Session 1: "Energy"
# Session 2: "IT"
# Session 3: "Financial"
# Session 4: "Healthcare"
# ── ↑↑↑ CHANGE THIS EACH SESSION ↑↑↑ ─────────────────────

ALL_SECTORS    = ["Energy"]
TARGET_SECTORS = [s for s in ALL_SECTORS if s != SOURCE_SECTOR]

# ── ALL 20 YEARS
ALL_YEARS = {idx: 2005 + idx for idx in range(20)}

# Years already completed in the original 9-significant-year run
SIGNIFICANT_YEARS = {
    0 : 2005,
    3 : 2008,
    4 : 2009,
    6 : 2011,
    7 : 2012,
    10: 2015,
    15: 2020,
    16: 2021,
    17: 2022,
}

N_QUBITS    = 4
N_LAYERS    = 2
N_STOCKS    = 15
HIDDEN_DIM  = 4
WINDOW_SIZE = 5
BATCH_SIZE  = 16
N_EPOCHS    = 10
LR          = 0.01

n_new_years = len(ALL_YEARS) - len(SIGNIFICANT_YEARS)
print(f"✅ Source: {SOURCE_SECTOR} → {TARGET_SECTORS}")
print(f"   Total years      : {len(ALL_YEARS)} (full 2005-2024)")
print(f"   Already trained  : {len(SIGNIFICANT_YEARS)} significant years")
print(f"   New years to run : {n_new_years}")
print(f"   New models       : {len(TARGET_SECTORS) * n_new_years}")

✅ Source: IT → ['Energy']
   Total years      : 20 (full 2005-2024)
   Already trained  : 9 significant years
   New years to run : 11
   New models       : 11


In [ ]:
# ── Quantum device ─────────────────────────────────────────
try:
    dev   = qml.device("lightning.qubit", wires=N_QUBITS)
    DIFF_METHOD = "adjoint"
    print("✅ Using lightning.qubit (fast)")
except Exception:
    dev   = qml.device("default.qubit", wires=N_QUBITS)
    DIFF_METHOD = "backprop"
    print("⚠️ Using default.qubit (slower)")


def vqc(inputs, weights):
    for i in range(N_QUBITS):
        qml.RY(inputs[i], wires=i)
    for i in range(N_QUBITS - 1):
        qml.CNOT(wires=[i, i + 1])
    qml.CNOT(wires=[N_QUBITS - 1, 0])
    for l in range(N_LAYERS):
        for i in range(N_QUBITS):
            qml.RY(weights[l, i, 0], wires=i)
            qml.RZ(weights[l, i, 1], wires=i)
        for i in range(N_QUBITS - 1):
            qml.CNOT(wires=[i, i + 1])
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]

qnode = qml.QNode(vqc, dev, interface="torch",
                  diff_method=DIFF_METHOD)

print("✅ Quantum circuit ready")

✅ Using lightning.qubit (fast)
✅ Quantum circuit ready


In [ ]:
# ── Cross-Sector qLSTM Cell ────────────────────────────────
class CrossSectorCell(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden_dim        = HIDDEN_DIM
        self.input_proj        = nn.Linear(N_STOCKS + HIDDEN_DIM, N_QUBITS)
        self.weights_forget    = nn.Parameter(torch.randn(N_LAYERS, N_QUBITS, 2) * 0.1)
        self.weights_input     = nn.Parameter(torch.randn(N_LAYERS, N_QUBITS, 2) * 0.1)
        self.weights_candidate = nn.Parameter(torch.randn(N_LAYERS, N_QUBITS, 2) * 0.1)
        self.weights_output    = nn.Parameter(torch.randn(N_LAYERS, N_QUBITS, 2) * 0.1)
        self.output_proj       = nn.Linear(HIDDEN_DIM, N_STOCKS)

    def forward_gate(self, projected, weights):
        results = []
        for b in range(projected.shape[0]):
            out = qnode(projected[b], weights)
            out_tensor = out.float() if isinstance(out, torch.Tensor) \
                         else torch.stack(out).float()
            results.append(out_tensor)
        return torch.stack(results)

    def forward(self, x_t, h_t, c_t):
        combined  = torch.cat([x_t, h_t], dim=-1)
        projected = torch.tanh(self.input_proj(combined)) * torch.pi
        f_t = torch.sigmoid(self.forward_gate(projected, self.weights_forget))
        i_t = torch.sigmoid(self.forward_gate(projected, self.weights_input))
        g_t = torch.tanh(self.forward_gate(projected,    self.weights_candidate))
        o_t = torch.sigmoid(self.forward_gate(projected, self.weights_output))
        c_new = f_t * c_t + i_t * g_t
        h_new = o_t * torch.tanh(c_new)
        return h_new, c_new, self.output_proj(h_new)


In [ ]:
# ── Cross-Sector Autoencoder with Dense Bottleneck ─────────
class CrossSectorModelDense(nn.Module):
    """
    Cross-sector qLSTM with dense bottleneck.
    Input  : source sector (15 stocks)
    Target : target sector (15 stocks)
    W extracted from self.dense.weight (Tuhin et al. method)
    """
    def __init__(self):
        super().__init__()
        self.hidden_dim = HIDDEN_DIM
        self.cell  = CrossSectorCell()
        # Dense bottleneck — W extracted directly from here
        self.dense = nn.Linear(N_STOCKS, N_STOCKS, bias=False)

    def forward(self, x):
        h_t = torch.zeros(x.shape[0], self.hidden_dim)
        c_t = torch.zeros(x.shape[0], self.hidden_dim)
        recons = []
        for t in range(x.shape[1]):
            h_t, c_t, recon = self.cell(x[:, t, :], h_t, c_t)
            recon = self.dense(recon)    # ← dense bottleneck
            recons.append(recon)
        return torch.stack(recons, dim=1)

print("✅ Cross-sector model with dense bottleneck defined")

✅ Cross-sector model with dense bottleneck defined


In [ ]:
# ── Helper functions ───────────────────────────────────────
def load_year(sector, year_idx):
    path = os.path.join(DATA_DIR, sector, f"year_{year_idx:02d}.csv")
    df   = pd.read_csv(path)
    arr  = df.drop(columns=["window_id","day"]).values.reshape(
               -1, WINDOW_SIZE, N_STOCKS)
    return torch.tensor(arr, dtype=torch.float32)


def get_model_path(source, target, year_idx):
    pair = f"{source}_to_{target}"
    return os.path.join(MODEL_DIR, pair, f"year_{year_idx:02d}.pt")


def is_trained(source, target, year_idx):
    return os.path.exists(get_model_path(source, target, year_idx))


def train_one(source, target, year_idx):
    """
    Trains one cross-sector model with checkpointing.
    Tracks the best epoch (lowest loss) and saves BOTH:
      - the final-epoch model state (for continuity)
      - the best-epoch model state (for weight extraction)
    Returns: best_model, final_model, losses, best_epoch_idx
    """
    x_src = load_year(source, year_idx)
    x_tgt = load_year(target, year_idx)

    loader  = DataLoader(TensorDataset(x_src, x_tgt),
                         batch_size=BATCH_SIZE, shuffle=True)
    model   = CrossSectorModelDense()
    optim   = torch.optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.MSELoss()
    losses  = []

    best_loss  = float("inf")
    best_state = None
    best_epoch = -1

    for epoch in range(N_EPOCHS):
        epoch_loss, n = 0.0, 0
        for src_b, tgt_b in loader:
            optim.zero_grad()
            recon = model(src_b)
            loss  = loss_fn(recon, tgt_b)
            loss.backward()
            optim.step()
            epoch_loss += loss.item()
            n += 1
        avg = epoch_loss / n
        losses.append(avg)
        print(f"      Epoch {epoch+1:02d}/{N_EPOCHS}  loss={avg:.4f}")

        # ── Checkpointing: track best epoch ────────────────
        if avg < best_loss:
            best_loss  = avg
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())

    return model, best_state, losses, best_epoch, best_loss

print("✅ Helper functions ready (with checkpointing)")

✅ Helper functions ready (with checkpointing)


In [ ]:
# ── Main training loop ─────────────────────────────────────
print("=" * 60)
print(f" Cross-Sector Dense Training — Full 20-Year Extension")
print(f" Source : {SOURCE_SECTOR}")
print(f" Targets: {TARGET_SECTORS}")
print(f" Years  : ALL 20 years (2005-2024)")
print(f" Models : {len(TARGET_SECTORS) * len(ALL_YEARS)} total")
print("=" * 60)

total_trained = 0
total_skipped = 0
total_start   = time.time()

for target in TARGET_SECTORS:
    print(f"\n{'='*60}")
    print(f" {SOURCE_SECTOR} → {target}")
    print(f"{'='*60}")

    pair      = f"{SOURCE_SECTOR}_to_{target}"
    pair_dir  = os.path.join(MODEL_DIR, pair)
    loss_dir  = os.path.join(LOSS_DIR,  pair)
    os.makedirs(pair_dir, exist_ok=True)
    os.makedirs(loss_dir, exist_ok=True)

    for year_idx, year_label in ALL_YEARS.items():
        tag = "significant" if year_idx in SIGNIFICANT_YEARS else "extension"
        print(f"\n  ── Year {year_idx:02d} ({year_label}) [{tag}] ──────────────")

        if is_trained(SOURCE_SECTOR, target, year_idx):
            print(f"  Already trained ✅ skipping")
            total_skipped += 1
            continue

        year_start = time.time()
        final_model, best_state, losses, best_epoch, best_loss = \
            train_one(SOURCE_SECTOR, target, year_idx)
        elapsed = time.time() - year_start

        # ── Save BEST-epoch model (used for weight extraction) ──
        save_path = get_model_path(SOURCE_SECTOR, target, year_idx)
        torch.save(best_state, save_path)

        # ── Save final-epoch model separately (for reference) ──
        final_dir  = os.path.join(MODEL_DIR, pair, "final_epoch")
        os.makedirs(final_dir, exist_ok=True)
        final_path = os.path.join(final_dir, f"year_{year_idx:02d}.pt")
        torch.save(final_model.state_dict(), final_path)

        # ── Save losses + checkpoint metadata ──────────────────
        pd.DataFrame({
            "epoch": range(1, N_EPOCHS + 1),
            "loss" : losses
        }).to_csv(os.path.join(loss_dir,
                  f"year_{year_idx:02d}_losses.csv"), index=False)

        total_trained += 1
        print(f"  ✅ Saved (best epoch {best_epoch+1}) → {save_path}")
        print(f"  Loss     : {losses[0]:.4f} → {losses[-1]:.4f}  "
              f"(best={best_loss:.4f} @ epoch {best_epoch+1})")
        print(f"  Time     : {elapsed:.1f}s")

total_elapsed = time.time() - total_start
print(f"\n{'='*60}")
print(f" ✅ Training complete")
print(f"    Trained : {total_trained}")
print(f"    Skipped : {total_skipped} (already done)")
print(f"    Time    : {total_elapsed/60:.1f} min")
print(f"{'='*60}")

 Cross-Sector Dense Training — Full 20-Year Extension
 Source : IT
 Targets: ['Energy']
 Years  : ALL 20 years (2005-2024)
 Models : 20 total

 IT → Energy

  ── Year 00 (2005) [significant] ──────────────
  Already trained ✅ skipping

  ── Year 01 (2006) [extension] ──────────────
      Epoch 01/10  loss=0.5105
      Epoch 02/10  loss=0.4975
      Epoch 03/10  loss=0.4547
      Epoch 04/10  loss=0.4254
      Epoch 05/10  loss=0.4095
      Epoch 06/10  loss=0.3964
      Epoch 07/10  loss=0.3939
      Epoch 08/10  loss=0.3891
      Epoch 09/10  loss=0.3854
      Epoch 10/10  loss=0.3786
  ✅ Saved (best epoch 10) → /content/drive/MyDrive/MSC Thesis/CrossSectorModels_Dense/IT_to_Energy/year_01.pt
  Loss     : 0.5105 → 0.3786  (best=0.3786 @ epoch 10)
  Time     : 409.4s

  ── Year 02 (2007) [extension] ──────────────
      Epoch 01/10  loss=0.5970
      Epoch 02/10  loss=0.5442
      Epoch 03/10  loss=0.4764
      Epoch 04/10  loss=0.4405
      Epoch 05/10  loss=0.4349
      Epoch 06/10  

In [ ]:
from google.colab import drive
drive.mount('/content/drive')